# Cogni Inference Notebook

This notebook mirrors `scripts/run_inference.py` for local inference checks.

In [ ]:
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B"
PROMPT = (
    "AP Language sample argument: The school board should remove all long-form essays "
    "because students mostly consume short social posts, so essay writing is no longer useful. "
    "Analyze whether this argument is logically sound."
)
OUTPUT_DIR = Path("outputs/inference_demo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "mps" else torch.float32
print("device:", DEVICE)


In [ ]:
load_start = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=DTYPE,
)
model.to(DEVICE)
model.eval()
load_time = time.perf_counter() - load_start
print(f"load time: {load_time:.4f}s")


In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are Cogni, an educational assistant for logical reasoning.",
    },
    {"role": "user", "content": PROMPT},
]
prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

encoded = tokenizer(prompt_text, return_tensors="pt")
encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

infer_start = time.perf_counter()
with torch.inference_mode():
    output_ids = model.generate(
        **encoded,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
infer_time = time.perf_counter() - infer_start

generated_ids = output_ids[0][encoded["input_ids"].shape[-1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print("model name:", MODEL_ID)
print("device:", DEVICE)
print(f"inference time: {infer_time:.4f}s")
print("generated response:")
print(response)


In [ ]:
(OUTPUT_DIR / "response.txt").write_text(response + "\n", encoding="utf-8")
(OUTPUT_DIR / "report.md").write_text(
    "\n".join(
        [
            "# Inference Verification Report",
            "",
            f"- model version: `{MODEL_ID}`",
            f"- device used: `{DEVICE}`",
            f"- load time (s): `{load_time:.4f}`",
            f"- inference time (s): `{infer_time:.4f}`",
            "",
            "## generated response",
            "",
            response,
            "",
        ]
    ),
    encoding="utf-8",
)
print("Saved outputs to", OUTPUT_DIR)
